# Experiment 4B (`DYN_REQUEST_GUARD`) Full-Budget 16K Rollout (Colab A100 80GB / 40GB)

Strictly paired with Experiment 2 `TEST 100` (`eval_multi_turn/EXPERIMENT_CONTRACT.md`):
- **3 Open-Weight Reasoning Models (BF16, exact pinned HF commits matching Exp 2)**:
  - `Qwen/Qwen3-8B` @ `b968826d9c46dd6066d109eabc6255188de91218`
  - `zai-org/GLM-Z1-9B-0414` @ `b221b06fefb23ca320922cf6e68ab5f2fb82de81`
  - `Qwen/Qwen3-14B` @ `40c069824f4251a91eefaf281ebe4c544efd3e18`
- **Protocol**: `DYN_REQUEST_GUARD` only (`reps=3`, `300` trajectories / `1,152` calls per model = **`900` trajectories / `3,456` calls total**)
- **Paired Controls**: Identical `frozen_test_100.jsonl`, `disclosure_plan_test100.jsonl` (same disclosure orders, pre-frozen `DYN_REQUEST_GUARD` seeds), `max_tokens=16384`, `max_model_len=32768`
- **Isolated Outputs**: Writes to `eval_multi_turn/runs/runs_16k/test100_{tag}_guard_16384.jsonl` so archived 5-protocol Exp 2 rollouts remain untouched

In [ ]:
# 1. Mount Drive, Unzip Code Only (runs/ excluded), chdir, & Check GPU
import os
import torch
from google.colab import drive

drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/cscwd_v2_colab_exp4b.zip -d /content/drive/MyDrive/
WORK_DIR = '/content/drive/MyDrive/cscwd_v2'
os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())
!nvidia-smi
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU VRAM: {vram:.2f} GiB')

In [ ]:
# 2. Install runtime dependencies & verify offline self-tests
import os
os.chdir('/content/drive/MyDrive/cscwd_v2')
!pip -q install -U vllm huggingface_hub openai
print('Confirmed CWD:', os.getcwd())
!python eval_multi_turn/protocol_utils.py --self-test
!python eval_multi_turn/rollout_vllm.py --mock-test
!python eval_multi_turn/analyze_experiments.py --mock-test

In [ ]:
# 3. [EXP 4B TEST 100 16K ROLLOUT] Run DYN_REQUEST_GUARD for all 3 models (reps=3) into isolated *_guard_16384.jsonl
import glob
import json
import os

os.chdir('/content/drive/MyDrive/cscwd_v2')
MODELS = [
    ('Qwen/Qwen3-8B', 'b968826d9c46dd6066d109eabc6255188de91218'),
    ('zai-org/GLM-Z1-9B-0414', 'b221b06fefb23ca320922cf6e68ab5f2fb82de81'),
    ('Qwen/Qwen3-14B', '40c069824f4251a91eefaf281ebe4c544efd3e18'),
]
for m, rev in MODELS:
    tag = m.split('/')[-1].lower().replace('-', '_')
    out_path = f'eval_multi_turn/runs/runs_16k/test100_{tag}_guard_16384.jsonl'
    print(f'\n================ Running Exp 4B DYN_REQUEST_GUARD: {m} ({rev[:8]}) ================')
    !python eval_multi_turn/rollout_vllm.py \
      --split TEST \
      --plan eval_multi_turn/disclosure_plan_test100.jsonl \
      --model {m} \
      --revision {rev} \
      --protocols DYN_REQUEST_GUARD \
      --reps 3 \
      --max-tokens 16384 \
      --max-model-len 32768 \
      --output {out_path} \
      --resume

print('\n=== EXP 4B (DYN_REQUEST_GUARD) TEST 100 16K SUMMARY ===')
for p in sorted(glob.glob('eval_multi_turn/runs/runs_16k/test100_*_guard_16384_meta.json')):
    meta = json.load(open(p, encoding='utf-8'))
    by_p = meta['by_protocol']['DYN_REQUEST_GUARD']
    rev = meta['provenance']['hf_model_revision'][:8]
    print(
        f"{os.path.basename(p):44s} | rev={rev} | done={by_p['completed']:3d}/300 "
        f"| turns={by_p['turns']:4d}/1152 | empty_trunc={by_p['empty_truncated_turns']:2d} "
        f"| clamp={by_p['budget_clamped_turns']:2d} | {meta['elapsed_sec']}s"
    )

In [ ]:
# 4. Auto-release A100 runtime immediately after Cell 3 finishes to save compute units
from google.colab import runtime
runtime.unassign()

In [ ]:
# 5. Local Dual-LLM Judging & Paired Exp 4B Analysis Reference
# Download these 6 files from Drive eval_multi_turn/runs/runs_16k/ to local runs/runs_16k/:
#   - test100_qwen3_8b_guard_16384.jsonl       (+ _meta.json)
#   - test100_glm_z1_9b_0414_guard_16384.jsonl (+ _meta.json)
#   - test100_qwen3_14b_guard_16384.jsonl      (+ _meta.json)
# Then run local Dual-LLM judging (Final Y + Scope C + Explicit Repair) & paired bootstrap analysis!